# 自己的圖片推論

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/08-own-images/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.3.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
from pathlib import Path
import numpy as np
from PIL import Image
import torch
from miniyolo.data import ShapeDataset, collate
from miniyolo.geometry import letterbox, undo_letterbox
from miniyolo.models import GridDetector
from miniyolo.targets import build_targets
from miniyolo.losses import grid_loss
from miniyolo.inference import decode_grid
from scripts.detect_image import load_grid_checkpoint, detect_image


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    folder = Path("artifacts/lesson-08-own-images")
    folder.mkdir(parents=True, exist_ok=True)
    pixels = np.zeros((80,120,3),dtype=np.uint8)
    pixels[10:30,20:60,0] = 255
    path = folder/'my_image.png'
    Image.fromarray(pixels).save(path)
    with Image.open(path) as opened:
        array = np.asarray(opened.convert('RGB')).copy()
    image = torch.from_numpy(array).permute(2,0,1).float()/255
    original_box = torch.tensor([[20.,10.,60.,30.]])
    prepared, input_boxes, meta = letterbox(image,original_box,size=64)
    assert prepared.shape == (3,64,64)
    restored = undo_letterbox(input_boxes,meta)
    assert torch.allclose(restored,original_box,atol=1e-4)
    print('original CHW',tuple(image.shape),'input box',input_boxes.tolist())
    print('metadata',meta,'roundtrip',restored.tolist())
    # Actually update parameters, save them, and apply the saved state_dict.
    dataset = ShapeDataset(n=4,size=64,seed=7,max_objects=1,allow_empty=False)
    train_images, train_anns = collate([dataset[i] for i in range(4)])
    model = GridDetector(num_classes=2,grid_size=4,width=8)
    optimizer = torch.optim.Adam(model.parameters(),lr=.01)
    before = next(model.parameters()).detach().clone()
    target = build_targets(train_anns,4,64,2)
    model.train()
    for _ in range(3):
        optimizer.zero_grad(set_to_none=True)
        loss = grid_loss(model(train_images),target)['total']
        loss.backward()
        optimizer.step()
    assert not torch.equal(before,next(model.parameters()).detach())
    config = {'image_size':64,'grid_size':4,'width':8,'num_classes':2,
              'score_threshold':.05,'nms_iou':.5}
    checkpoint_path = folder/'checkpoint.pt'
    torch.save({'model_state_dict':model.state_dict(),'config':config,
                'class_names':['red rectangle','blue rectangle'],'steps_completed':3},checkpoint_path)
    reloaded, loaded_config, classes = load_grid_checkpoint(checkpoint_path)
    model.eval()
    with torch.inference_mode():
        assert torch.equal(model(prepared[None]),reloaded(prepared[None]))
    assert loaded_config == config and len(classes) == 2
    output_path = folder/'prediction.png'
    report = detect_image(path,checkpoint_path,output_path)
    assert output_path.is_file() and output_path.with_suffix('.json').is_file()
    with Image.open(output_path) as saved:
        assert saved.size == (120,80)
    print('3-step checkpoint safely reloaded: exact logits; original-space box count',len(report['boxes']))
    print('annotated PNG and JSON saved:', output_path, output_path.with_suffix('.json'))
    print('pipeline evidence only, not photo detection quality')
    bad_path = folder/'wrong-class-count.pt'
    torch.save({'model_state_dict':model.state_dict(),'config':config,
                'class_names':['red','blue','yellow']},bad_path)
    try:
        load_grid_checkpoint(bad_path)
    except ValueError:
        print('class/config mismatch rejected')
    else:
        raise AssertionError('wrong class count accepted')
    target = build_targets([{'boxes':input_boxes,'labels':torch.tensor([0])}],4,64,2)
    fixture = torch.zeros(1,4,4,7)
    fixture[...,4] = -20
    b,y,x = target['positive'].nonzero()[0].tolist()
    fixture[b,y,x,:4] = torch.logit(target['box'][b,y,x])
    fixture[b,y,x,4] = 10
    fixture[b,y,x,5:] = torch.tensor([10.,-10.])
    known = decode_grid(fixture,64,.25,.5)[0]
    known_original = undo_letterbox(known['boxes'],meta)
    assert torch.allclose(known_original,original_box,atol=1e-3)
    print('artificial known box restored',known_original.tolist())

if __name__ == '__main__':
    main()


original CHW (3, 80, 120) input box [[10.666666984558105, 15.375, 32.0, 26.125]]
metadata {'original_size': (80, 120), 'scale': 0.5333333333333333, 'scale_xy': (0.5333333333333333, 0.5375), 'padding': (0, 10), 'resized_size': (43, 64)} roundtrip [[20.0, 10.0, 59.999996185302734, 29.999998092651367]]
3-step checkpoint safely reloaded: exact logits; original-space box count 16
annotated PNG and JSON saved: artifacts/lesson-08-own-images/prediction.png artifacts/lesson-08-own-images/prediction.json
pipeline evidence only, not photo detection quality
class/config mismatch rejected
artificial known box restored [[20.000001907348633, 10.0, 59.999996185302734, 29.999998092651367]]
